# 01 - Raw Data Audit

Audit raw sources, validate schemas, and record data quality notes.


Before analyzing churn, we need to validate whether the raw winer DTC data is reliable enough to support retention analysis.

This notebook audits:
- Table structure
- Row counts
- Primary key uniquness
- Foreign key relationships
- Missing values
- Date ranges
- Member age elligibility
- Basic source-system quality issues

The goal for this analysis is to understand whether the source data can answer why members churn.

In [37]:
# Load libraries and data

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

RAW_DIR = Path('../data/raw')
DIM_DIR = Path('../data/dim')



raw_files = {
    'raw_club_releases': 'raw_club_releases.csv',
    'raw_email_events': 'raw_email_events.csv',
    'raw_event_attendance': 'raw_event_attendance.csv',
    'raw_members': 'raw_members.csv',
    'raw_membership_events': 'raw_membership_events.csv',
    'raw_order_items': 'raw_order_items.csv',
    'raw_orders': 'raw_orders.csv',
    'raw_release_wines': 'raw_release_wines.csv',
    'raw_tasting_room_visits': 'raw_tasting_room_visits.csv',
    'raw_wines': 'raw_wines.csv'
}

dim_files = {
   'dim_acquisition_sources': 'dim_acquisition_sources.csv',
   'dim_club_tiers': 'dim_club_tiers.csv',
   'dim_dates': 'dim_dates.csv',
   'dim_members': 'dim_members.csv',
   'dim_releases': 'dim_releases.csv',
   'dim_wines': 'dim_wines.csv'
}

raw = {
    name:pd.read_csv(RAW_DIR/file)
    for name, file in raw_files.items()
}

dims = {
    name:pd.read_csv(DIM_DIR/file)
    for name, file in dim_files.items()
}


In [38]:
# Validate Row Counts

row_counts = []

for name, df in raw.items():
    row_counts.append({
        'table_name': name,
        'layer': 'raw',
        'row_count': len(df),
        'column_count': df.shape[1]
    })

for name, df in dims.items():
    row_counts.append({
        'table_name': name,
        'layer': 'dimension',
        'row_count': len(df),
        'column_count': df.shape[1]
    })

row_counts_df = pd.DataFrame(row_counts).sort_values(['layer', 'table_name'])
row_counts_df

,table_name,layer,row_count,column_count
10,dim_acquisition_sources,dimension,8,7
11,dim_club_tiers,dimension,4,10
12,dim_dates,dimension,1826,13
13,dim_members,dimension,4200,17
14,dim_releases,dimension,20,16
15,dim_wines,dimension,55,16
0,raw_club_releases,raw,20,12
1,raw_email_events,raw,223836,9
2,raw_event_attendance,raw,9634,10
3,raw_members,raw,4200,15


The dataset contains enough behavioral depth to support retention analysis and analyze churn as a relationship and behavior problem.

In [39]:
# Inspect schemas

for name, df in raw.items():
    print(f'\n{name}')
    print('-' * len(name))
    print(df.dtypes)


raw_club_releases
-----------------
release_id                 object
release_name               object
release_year                int64
release_season             object
release_open_date          object
release_ship_date          object
customization_deadline     object
default_2_bottle_value      int64
default_4_bottle_value      int64
default_6_bottle_value      int64
default_12_bottle_value     int64
release_theme              object
dtype: object

raw_email_events
----------------
email_event_id    object
member_id         object
campaign_id       object
send_date         object
email_type        object
event_type        object
link_category     object
release_id        object
device_type       object
dtype: object

raw_event_attendance
--------------------
event_attendance_id          object
member_id                    object
event_date                   object
event_name                   object
event_type                   object
ticket_value                  int64
guest_co

*** will need to change date fields to date format before building fact tables ***

In [40]:
# Drop Unnecessary Columns

# Columns to drop from raw tables
raw_columns_to_drop = {
    "raw_members": [
        "age_verified_date",
        "age_verification_method",
        "age_verified_flag",
        "created_at",
        "updated_at"
    ],
    "raw_membership_events": [
        "created_at"
    ],
    "raw_email_events": [
        "created_at"
    ],
    "raw_tasting_room_visits": [
        "created_at"
    ],
    "raw_event_attendance": [
        "created_at"
    ],
    "raw_wines": [
        "sellout_date",
        "created_at"
    ],
    "raw_club_releases": [
        "created_at"
    ],
    "raw_release_wines": [
        "created_at"
    ]
}

# Columns to drop from dimension tables
dim_columns_to_drop = {
    "dim_members": [
        "age_verified_date",
        "age_verification_method",
        "age_verified_flag",
        "created_at",
        "updated_at"
    ],
    "dim_wines": [
        "sellout_date"
    ],
    "dim_releases": [
        "created_at"
    ]
}

# Drop columns safely
for table_name, columns in raw_columns_to_drop.items():
    raw[table_name] = raw[table_name].drop(
        columns=columns,
        errors="ignore"
    )

for table_name, columns in dim_columns_to_drop.items():
    dims[table_name] = dims[table_name].drop(
        columns=columns,
        errors="ignore"
    )

In [41]:
# Confirm dropped columns were removed

for table_name, columns in raw_columns_to_drop.items():
    remaining_cols = [col for col in columns if col in raw[table_name].columns]
    print(f"{table_name}: {remaining_cols}")

for table_name, columns in dim_columns_to_drop.items():
    remaining_cols = [col for col in columns if col in dims[table_name].columns]
    print(f"{table_name}: {remaining_cols}")

raw_members: []
raw_membership_events: []
raw_email_events: []
raw_tasting_room_visits: []
raw_event_attendance: []
raw_wines: []
raw_club_releases: []
raw_release_wines: []
dim_members: []
dim_wines: []
dim_releases: []


In [42]:
# Save updaed datasets to our df

for table_name, df in raw.items():
    df.to_csv(RAW_DIR / f"{table_name}.csv", index=False)

for table_name, df in dims.items():
    df.to_csv(DIM_DIR / f"{table_name}.csv", index=False)

In [43]:
# Parse Key Dates

date_columns = {
    'raw_members': ['join_date', 'birth_date'],
    'raw_orders': ['order_date'],
    'raw_order_items': ['created_at'],
    'raw_membership_events': ['event_date'],
    'raw_tasting_room_visits': ['visit_date'],
    'raw_event_attendance': ['event_date'],
    'raw_wines': ['release_date'],
    'raw_club_releases': ['release_open_date', 'release_ship_date', 'customization_deadline']
}

for table_name, cols in date_columns.items():
    for col in cols:
        if col in raw[table_name].columns:
            raw[table_name][col] = pd.to_datetime(raw[table_name][col], errors='coerce')

In [44]:
# Primary Key Checks
# Valid primary keys should not have any nulls, duplicates, one unique identifier per row


primary_keys = {
    'raw_members': 'member_id',
    'raw_orders': 'order_id',
    'raw_order_items': 'order_item_id',
    'raw_membership_events': 'membership_event_id',
    'raw_email_events': 'email_event_id',
    'raw_tasting_room_visits': 'visit_id',
    'raw_event_attendance': 'event_attendance_id',
    'raw_wines': 'wine_sku',
    'raw_club_releases': 'release_id',
    'raw_release_wines': 'release_wine_id'
    }

pk_checks = []

for table_name, pk in primary_keys.items():
    df = raw[table_name]

    pk_checks.append({
        'table_name': table_name,
        'primary_key': pk,
        'rows': len(df),
        'unique_keys': df[pk].nunique(),
        'null_keys': df[pk].isnull().sum(),
        'duplicate_keys': len(df) - df[pk].nunique()
    })

    pk_checks_df = pd.DataFrame(pk_checks)

    pk_checks_df.sort_values('table_name')


In [48]:
# Foreign Key Checks
# confirm child keys can connect back to their parent keys
# This matters because the retention analysis depends on stitching together behavior across:
# member profiles, orders, wine purchases, membership lifecycle events, email engagement, tasting room visits, event attendance, and club releases

fk_checks = []

def check_fk(child_table, child_key, parent_table, parent_key):
    child = raw[child_table]
    parent = raw[parent_table]

    missing_count = (
        ~child[child_key].dropna().isin(parent[parent_key])
    ).sum()

    missing_rate = missing_count / child[child_key].dropna().shape[0]

    return{
        'child_table': child_table,
        'child_key': child_key,
        'parent_table': parent_table,
        'parent_key': parent_key,
        'child_row': len(child),
        'non_null_child_keys': child[child_key].notna().sum(),
        'missing_fk_count': missing_count,
        'missing_fk_rate': missing_rate
    }

fk_checks.append(check_fk("raw_orders", "member_id", "raw_members", "member_id"))
fk_checks.append(check_fk("raw_order_items", "order_id", "raw_orders", "order_id"))
fk_checks.append(check_fk("raw_order_items", "wine_sku", "raw_wines", "wine_sku"))
fk_checks.append(check_fk("raw_membership_events", "member_id", "raw_members", "member_id"))
fk_checks.append(check_fk("raw_email_events", "member_id", "raw_members", "member_id"))
fk_checks.append(check_fk("raw_tasting_room_visits", "member_id", "raw_members", "member_id"))
fk_checks.append(check_fk("raw_event_attendance", "member_id", "raw_members", "member_id"))
fk_checks.append(check_fk("raw_release_wines", "release_id", "raw_club_releases", "release_id"))
fk_checks.append(check_fk("raw_release_wines", "wine_sku", "raw_wines", "wine_sku"))

fk_checks_df = pd.DataFrame(fk_checks)

fk_checks_df

,child_table,child_key,parent_table,parent_key,child_row,non_null_child_keys,missing_fk_count,missing_fk_rate
0,raw_orders,member_id,raw_members,member_id,48953,48953,0,0.0
1,raw_order_items,order_id,raw_orders,order_id,176335,176335,0,0.0
2,raw_order_items,wine_sku,raw_wines,wine_sku,176335,176335,0,0.0
3,raw_membership_events,member_id,raw_members,member_id,43046,43046,0,0.0
4,raw_email_events,member_id,raw_members,member_id,223836,223836,0,0.0
5,raw_tasting_room_visits,member_id,raw_members,member_id,4736,4736,0,0.0
6,raw_event_attendance,member_id,raw_members,member_id,9634,9634,0,0.0
7,raw_release_wines,release_id,raw_club_releases,release_id,49,49,0,0.0
8,raw_release_wines,wine_sku,raw_wines,wine_sku,49,49,0,0.0


In [ ]:
# Age distribution across members


age_bins = [21, 30, 40, 50, 60, 70, 120]
age_labels = ["21-29", "30-39", "40-49", "50-59", "60-69", "70+"]

members["age_band_at_join"] = pd.cut(
    members["age_at_join"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

age_band_summary = (
    members
    .groupby("age_band_at_join", observed=False)
    .agg(
        members=("member_id", "nunique")
    )
    .reset_index()
)

age_band_summary["member_share"] = (
    age_band_summary["members"] / age_band_summary["members"].sum()
)

age_band_summary

,age_band_at_join,members,member_share
0,21-29,227,0.057629
1,30-39,562,0.142676
2,40-49,886,0.224930
3,50-59,1064,0.270119
4,60-69,870,0.220868
5,70+,330,0.083778


In [52]:
# Date range audit
# Confirm raw tables cover 5 years of membership data across key business dates 

date_audit = []

date_fields = {
    "raw_members": "join_date",
    "raw_orders": "order_date",
    "raw_membership_events": "event_date",
    "raw_email_events": "send_date",
    "raw_tasting_room_visits": "visit_date",
    "raw_event_attendance": "event_date",
    "raw_wines": "release_date",
    "raw_club_releases": "release_ship_date"
}

for table_name, date_col in date_fields.items():
    df = raw[table_name]
    
    date_audit.append({
        "table_name": table_name,
        "date_field": date_col,
        "min_date": df[date_col].min(),
        "max_date": df[date_col].max(),
        "null_dates": df[date_col].isna().sum(),
        "row_count": len(df)
    })

date_audit_df = pd.DataFrame(date_audit)

date_audit_df

/var/folders/kh/pp8y737s1tqfjqp2sjl22hbh0000gn/T/ipykernel_94632/459473302.py:29: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  date_audit_df = pd.DataFrame(date_audit)


,table_name,date_field,min_date,max_date,null_dates,row_count
0,raw_members,join_date,2021-01-01 00:00:00,2025-12-28 00:00:00,0,4200
1,raw_orders,order_date,2021-01-07 00:00:00,2025-12-30 00:00:00,0,48953
2,raw_membership_events,event_date,2021-01-01 00:00:00,2026-01-02 00:00:00,0,43046
3,raw_email_events,send_date,2021-02-14 07:03:00,2025-12-09 22:43:00,0,223836
4,raw_tasting_room_visits,visit_date,2020-12-31 00:00:00,2025-12-30 00:00:00,0,4736
5,raw_event_attendance,event_date,2021-03-08 00:00:00,2025-12-02 00:00:00,0,9634
6,raw_wines,release_date,2021-03-14 00:00:00,2025-11-19 00:00:00,0,55
7,raw_club_releases,release_ship_date,2021-03-14 00:00:00,2025-11-19 00:00:00,0,20


Raw datasets cover operating window from 2021 through 2025

This gives us enough historical depth to evaluate early member churn, first-year retention, multi-year loyalty, release-season churn patterns, and right-censored active members for survival analysis.

In [53]:
# Member Status Snapshot

member_status_summary = (
    raw["raw_members"]
    .groupby("member_status")
    .agg(
        members=("member_id", "nunique")
    )
    .reset_index()
)

member_status_summary["member_share"] = (
    member_status_summary["members"] / member_status_summary["members"].sum()
)

member_status_summary

,member_status,members,member_share
0,active,3749,0.892619
1,cancelled,451,0.107381


### Raw Data Audit Summary

The raw data tables are structurally ready for retention analysis.

The dataset includes the core source-system tables needed to analyze member churn:

- member profiles
- orders
- order items
- membership lifecycle events
- email engagement
- tasting room visits
- event attendance
- wine catalog data
- club release data
- release wine mappings

The primary key and foreign key checks confirm that the raw tables can be reliably joined across member, order, wine, release, and engagement activity.

The current CRM member status snapshot shows that 10.7% of members are cancelled and 89.3% are active.
